# GeoSPARQL 1.1 Tutorial

This tutorial showcases the newly introduced functions of the GeoSPARQL 1.1 standard which have been implemented in rdflib-geosparql.

In [30]:
from geosparql.geosparql import geosparql11

print("Total: "+str(len(geosparql11))+" functions")

Total: 33 functions
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


## Installing requirements

In [1]:
!pip install -r requirements.txt --break-system-packages
!pip install ipyleaflet --break-system-packages
!jupyter labextension install jupyter-leaflet

import rdflib
import shapely.geometry
from shapely import wkt
from shapely.ops import transform
from shapely.ops import unary_union
from rdflib import Graph, Literal
import os
import itertools
from shapely.testing import assert_geometries_equal
from geosparql.geosparql import LiteralUtils
from geosparql.geosparql_aggregates import processLiteralTypeToGeom
from IPython.display import display, HTML
import ipywidgets as W
from ipyleaflet import Map, WKTLayer, FullScreenControl, LayersControl, Popup#, Tooltip

mapcenter=(34.1,-83.2)
zoomlevel=10
GEO = "http://www.opengis.net/ont/geosparql#"
GEOF = "http://www.opengis.net/def/function/geosparql/"
GEOFEXT = "http://www.opengis.net/def/function/geosparql/ext/"

def getHTMLStringFromQueryResult(qres):
    res="<table><tr><th>Variable</th><th>Value</th></tr>"
    if qres is not None and len(qres.bindings)>0:
        resultlist=[{str(k): v for k, v in i.items()} for i in qres.bindings]
    for row in resultlist:
        for item in row:
            if isinstance(row[item],Literal) and row[item].datatype!=None:
                res+="<tr><td>"+str(item)+"</td><td><a href=\""+str(row[item].datatype)+"\">"+str(row[item]).strip().replace("<","&lt;").replace(">","&gt;").replace("\n","<br/>")+"</a></td></tr>"
            elif isinstance(row[item],URIRef):
                res+="<tr><td>"+str(item)+"</td><td><a href=\""+str(row[item])+"\">"+str(row[item]).strip()+"</a></td></tr>"
            else:
                res+="<tr><td>"+str(item)+"</td><td>"+str(row[item]).strip().replace("<","&lt;").replace(">","&gt;").replace("\n","<br/>")+"</td></tr>"
    res+="</table>"
    return res

def getMapFromWKTResult(qres,rows=[],lmap=None):
    layers=[]
    lastcentroid=shapely.geometry.Point(mapcenter[0],mapcenter[1])
    bboxes=[]
    if qres is not None and len(qres.bindings)>0:
        resultlist=[{str(k): v for k, v in i.items()} for i in qres.bindings]
        for row in rows:
            if resultlist!=None and row in resultlist[0]:
                popup="<h3>"+str(row)+"</h3><ul>"        
                for rowres in resultlist:
                    for item in rowres:
                        if isinstance(rowres[item],Literal) and rowres[item].datatype!=None:
                            popup+="<li><b>"+str(item)+":</b> <a href=\""+str(rowres[item].datatype)+"\">"+str(rowres[item]).strip()+"</a></li>"
                        elif isinstance(rowres[item],URIRef):
                            popup+="<li><b>"+str(item)+":</b> <a href=\""+str(rowres[item])+"\">"+str(rowres[item]).strip()+"</a></li>"
                        else:
                            popup+="<li><b>"+str(item)+":</b> "+str(rowres[item]).strip()+"</li>"
                popup+="</ul>"
                toprocess=resultlist[0][row].strip()
                if toprocess.startswith("<http"):
                    toprocess=toprocess[toprocess.find(">")+1:]
                geom = wkt.loads(toprocess)
                if not shapely.is_empty(geom):
                    lastcentroid=geom.centroid
                    nlayer=WKTLayer(name=str(row),wkt_string=geom.wkt,hover_style={"fillColor": "red"})
                    bounds=geom.bounds
                    bboxes.append(shapely.geometry.box(bounds[0],bounds[1],bounds[2],bounds[3]))
                    msg=W.HTML()
                    msg.value=popup
                    nlpopup=Popup(
                        location=[lastcentroid.y,lastcentroid.x],
                        child=msg,
                        close_button=True,
                        auto_close=False,
                        close_on_escape_key=False
                    )
                    nlayer.popup=msg
                    layers.append(nlayer)
    if lmap==None:
        maxbounds=unary_union(bboxes).bounds
        lmap=Map(center=(lastcentroid.y,lastcentroid.x), zoom=zoomlevel)
        control = FullScreenControl()
        lmap.add(control)
        control = LayersControl(position='topright')
        lmap.add(control)
    for lay in layers:
        lmap.add(lay)
    lmap.fit_bounds(((maxbounds[1],maxbounds[2]),(maxbounds[3],maxbounds[0])))
    return lmap     

def processQueryResults(qres,rows=[],lmap=None):
    display(HTML(getHTMLStringFromQueryResult(qres)))
    if rows!=[]:
        lmap=getMapFromWKTResult(qres,rows,lmap)
    return lmap

g=Graph()
g.parse("tests/testdata.ttl")
print(len(g))

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
(Deprecated) Installing extensions with the jupyter labextension install command is now deprecated and will be removed in a future major version of JupyterLab.

Users should manage prebuilt extensions with package managers like pip and conda, and extension authors are encouraged to distribute their extensions as prebuilt packages 
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:54: UserWarning: An error occurred.
  warnings.warn("An error occurred.")
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:55: UserWarning: PermissionError: [Errno 13] Permission denied: '/usr/local/share/jupyter/lab/extensions/jupyter-leaflet-0.20.0.tgz'
  warnings.warn(msg[-1].strip())
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:56: UserWarning: See the log file for details: /tmp/jupyterlab-debug-pg5cg8g_.log


## GeoSPARQL 1.1 Serialization Functions

### [geof:asDGGS](http://www.opengis.net/def/function/geosparql/asDGGS) Function

Retrieves a DGGS serialization of a GeoSPARQL geometry as [geo:dggsLiteral](http://www.opengis.net/ont/geosparql#dggsLiteral).

The example shows the DGGS literal with the H3 DGGS system.

In [2]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?dggs
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asDGGS(?aLiteral,"https://h3geo.org/res/7") as ?dggs)
}
"""),["aLiteral"],None)

Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:asGeoJSON](http://www.opengis.net/def/function/geosparql/asGeoJSON) Function

Retrieves a GeoJSON serialization of a GeoSPARQL geometry as [geo:geoJSONLiteral](http://www.opengis.net/ont/geosparql#geoJSONLiteral).

If the input geometry is not in the WGS84 coordinate system, the geometry will be converted to WGS84.

In [3]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?geojson
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asGeoJSON(?aLiteral) as ?geojson)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
geojson,"{""type"":""Polygon"",""coordinates"":[[[-83.6,34.1],[-83.2,34.1],[-83.2,34.5],[-83.6,34.5],[-83.6,34.1]]]}"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:asGML](http://www.opengis.net/def/function/geosparql/asGML) Function

Retrieves a GML serialization of a GeoSPARQL geometry as [geo:gmlLiteral](http://www.opengis.net/ont/geosparql#gmlLiteral).

In [4]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?gml
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asGML(?aLiteral) as ?gml)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:asKML](http://www.opengis.net/def/function/geosparql/asKML) Function

Retrieves a KML serialization of a GeoSPARQL geometry as [geo:kmlLiteral](http://www.opengis.net/ont/geosparql#kmlLiteral).

If the input geometry is not in the WGS84 coordinate system, the geometry will be converted to WGS84.

In [5]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?kml
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asKML(?aLiteral) as ?kml)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
kml,"<kml xmlns=""http://www.opengis.net/kml/2.2""><Placemark><kml:Polygon xmlns:kml=""http://www.opengis.net/kml/2.2""> <kml:outerBoundaryIs> <kml:LinearRing> <kml:coordinates>-83.6,34.1 -83.2,34.1 -83.2,34.5 -83.6,34.5 -83.6,34.1</kml:coordinates> </kml:LinearRing> </kml:outerBoundaryIs></kml:Polygon></Placemark></kml>"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:asWKT](http://www.opengis.net/def/function/geosparql/asWKT) Function

Retrieves a Well-Known Text serialization of a GeoSPARQL geometry as [geo:wktLiteral](http://www.opengis.net/ont/geosparql#wktLiteral).

In [6]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?wkt
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asGeoJSON ?aLiteral .
  BIND (geof:asWKT(?aLiteral) as ?wkt)
}
"""),["wkt"],None)

Variable,Value
aLiteral,"{""type"": ""Polygon"", ""coordinates"": [[[-83.6, 34.1], [-83.2, 34.1], [-83.2, 34.5], [-83.6, 34.5], [-83.6, 34.1]]]}"
wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

## Non-topological query functions

### [geof:area](http://www.opengis.net/def/function/geosparql/area) Function

Returns the area of a given geometry in the unit provided as [xsd:double](http://www.w3.org/2001/XMLSchema#double)

In [7]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?area
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:area(?aLiteral,"http://qudt.org/vocab/unit/AC"^^xsd:anyURI) as ?area)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
area,593079.4121828682


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:boundingCircle](http://www.opengis.net/def/function/geosparql/boundingCircle) Function

Calculates the minimum bounding circle around a given geometry and returns it in the literal format and CRS of the input geometry.

In [8]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bcirc
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:boundingCircle(?aLiteral) as ?bcirc)
}
"""),["aLiteral","bcirc"],None)

POLYGON ((-83.1171572875254 34.3, -83.12259203093559 34.24482012414341, -83.13868740702475 34.191760779970764, -83.16482487951615 34.14286100832258, -83.20000000000002 34.10000000000001, -83.24286100832259 34.06482487951614, -83.29176077997077 34.03868740702473, -83.34482012414342 34.02259203093558, -83.4 34.017157287525386, -83.45517987585659 34.02259203093558, -83.50823922002924 34.03868740702473, -83.55713899167742 34.06482487951614, -83.6 34.1, -83.63517512048386 34.14286100832258, -83.66131259297526 34.191760779970764, -83.67740796906442 34.24482012414341, -83.68284271247461 34.3, -83.67740796906442 34.355179875856585, -83.66131259297526 34.40823922002923, -83.63517512048386 34.45713899167741, -83.6 34.499999999999986, -83.55713899167742 34.535175120483856, -83.50823922002924 34.56131259297526, -83.45517987585659 34.57740796906442, -83.4 34.58284271247461, -83.34482012414342 34.57740796906442, -83.29176077997077 34.56131259297526, -83.24286100832259 34.535175120483856, -83.2000000

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
bcirc,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.1171572875254 34.3, -83.12259203093559 34.24482012414341, -83.13868740702475 34.191760779970764, -83.16482487951615 34.14286100832258, -83.20000000000002 34.10000000000001, -83.24286100832259 34.06482487951614, -83.29176077997077 34.03868740702473, -83.34482012414342 34.02259203093558, -83.4 34.017157287525386, -83.45517987585659 34.02259203093558, -83.50823922002924 34.03868740702473, -83.55713899167742 34.06482487951614, -83.6 34.1, -83.63517512048386 34.14286100832258, -83.66131259297526 34.191760779970764, -83.67740796906442 34.24482012414341, -83.68284271247461 34.3, -83.67740796906442 34.355179875856585, -83.66131259297526 34.40823922002923, -83.63517512048386 34.45713899167741, -83.6 34.499999999999986, -83.55713899167742 34.535175120483856, -83.50823922002924 34.56131259297526, -83.45517987585659 34.57740796906442, -83.4 34.58284271247461, -83.34482012414342 34.57740796906442, -83.29176077997077 34.56131259297526, -83.24286100832259 34.535175120483856, -83.20000000000002 34.49999999999999, -83.16482487951615 34.45713899167741, -83.13868740702475 34.40823922002923, -83.12259203093559 34.355179875856585, -83.1171572875254 34.3))"


Map(center=[34.3, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:centroid](http://www.opengis.net/def/function/geosparql/centroid) Function 🧊

Calculates the centroid of a given geometry. The centroid is returned in the literal format and CRS of the input geometry.

In [9]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX geofext: <"""+str(GEOFEXT)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?centroid ?centroid3d
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:centroid(?aLiteral) as ?centroid)
  BIND (geof:centroid("POLYGON Z ((-83.6 34.1 5.0, -83.2 34.1 5.0, -83.2 34.5 5.0, -83.6 34.5 5.0, -83.6 34.1 5.0))"^^geo:wktLiteral) as ?centroid3d)
}
"""),["aLiteral","centroid"],None)

POLYGON ((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
POLYGON Z ((-83.6 34.1 5, -83.2 34.1 5, -83.2 34.5 5, -83.6 34.5 5, -83.6 34.1 5))


Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
centroid,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POINT (-83.39999999999999 34.300000000000004)
centroid3d,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POINT Z (-83.39999999999999 34.300000000000004 5)


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:concaveHull](http://www.opengis.net/def/function/geosparql/concaveHull) Function

In [10]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?chull
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:concaveHull(?aLiteral) as ?chull)
}
"""),["aLiteral","chull"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
chull,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-83.2 34.1, -83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.1))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:coordinateDimension](http://www.opengis.net/def/function/geosparql/coordinateDimension) Function

Returns the coordinate dimension of a given geometry as [xsd:integer](http://www.w3.org/2001/XMLSchema#integer)

In [11]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?coordinateDimension
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:coordinateDimension(?literal) AS ?coordinateDimension)
}
"""),["literal"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
coordinateDimension,2


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:geometryN](http://www.opengis.net/def/function/geosparql/geometryN) Function

Returns the nth geometry of a given Multigeometry or GeometryCollection in the literal format and CRS of the input geometry

In [12]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?geomN
WHERE {
  BIND("MULTIPOLYGON (((1 1, 1 3, 3 3, 3 1, 1 1)), ((4 3, 6 3, 6 1, 4 1, 4 3)))"^^geo:wktLiteral AS ?aLiteral)
  BIND(geof:geometryN(?aLiteral,"1"^^xsd:integer) AS ?geomN)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"MULTIPOLYGON (((1 1, 1 3, 3 3, 3 1, 1 1)), ((4 3, 6 3, 6 1, 4 1, 4 3)))"
geomN,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((4 3, 6 3, 6 1, 4 1, 4 3))"


Map(center=[2.0, 3.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

### [geof:is3D](http://www.opengis.net/def/function/geosparql/is3D) Function

Indicates whether the given input geometry is threedimensional. The result is returned as a [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [13]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?n_wkt ?k_wkt ?isN3D ?isK3D {
<http://example.org/ApplicationSchema#NExactGeom> geo:asWKT ?n_wkt .
BIND(geof:is3D(?n_wkt) AS ?isN3D)
<http://example.org/ApplicationSchema#KExactGeom> geo:asWKT ?k_wkt .
BIND(geof:is3D(?k_wkt) AS ?isK3D)
}
"""),["n_wkt","k_wkt"],None)

Variable,Value
n_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon Z((-77.089005 38.913574 1,-77.029953 38.913574 2,-77.029953 38.886321 2,-77.089005 38.886321 1,-77.089005 38.913574 1))"
k_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-77.089005 38.913574,-77.029953 38.913574,-77.029953 38.886321,-77.089005 38.886321,-77.089005 38.913574))"


Map(center=[38.8999475, -77.05947900000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

### [geof:isEmpty](http://www.opengis.net/def/function/geosparql/isEmpty) Function

Indicates whether a gvien geometry is empty. The result is returned as [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [14]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?i_wkt ?k_wkt (xsd:boolean(?isIEmptyr) as ?isIEmpty) (xsd:boolean(?isKEmptyr) as ?isKEmpty) {
    <http://example.org/ApplicationSchema#IExactGeom> geo:asWKT ?i_wkt .
    BIND(geof:isEmpty(?i_wkt) AS ?isIEmptyr)
     <http://example.org/ApplicationSchema#KExactGeom> geo:asWKT ?k_wkt .
    BIND(geof:isEmpty(?k_wkt) AS ?isKEmptyr)
}
"""),["i_wkt","k_wkt"],None)

Variable,Value
i_wkt,LINESTRING EMPTY
k_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-77.089005 38.913574,-77.029953 38.913574,-77.029953 38.886321,-77.089005 38.886321,-77.089005 38.913574))"
isIEmpty,true
isKEmpty,false


Map(center=[38.8999475, -77.05947900000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

### [geof:isMeasured](http://www.opengis.net/def/function/geosparql/isMeasured) Function

Indicates the presence or absence of a measurement coordinate in a given geometry. The result is returned as [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [15]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?e_wkt ?l_wkt ?isEMeasured ?isLMeasured {
    BIND("POINT (1 2)"^^geo:wktLiteral AS ?e_wkt)
    BIND("POINT M (1 2 3)"^^geo:wktLiteral AS ?l_wkt)
    BIND(geof:isMeasured(?e_wkt) AS ?isEMeasured)
    BIND(geof:isMeasured(?l_wkt) AS ?isLMeasured)
}
"""),["e_wkt","l_wkt"],None)

Variable,Value
e_wkt,POINT (1 2)
l_wkt,POINT M (1 2 3)
isEMeasured,false
isLMeasured,true


Map(center=[2.0, 1.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:727: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


### [geof:isSimple](http://www.opengis.net/def/function/geosparql/isSimple) Function

Indicates whether the given geometry is simple. The result is returned as [xsd:boolean](http://www.w3.org/2001/XMLSchema#boolean)

In [16]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?o_wkt ?isASimple ?isOSimple {
<http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
BIND(geof:isSimple(?a_wkt) AS ?isASimple)
<http://example.org/ApplicationSchema#OExactGeom> geo:asWKT ?o_wkt .
BIND(geof:isSimple(?o_wkt) AS ?isOSimple)
}
"""),["a_wkt","o_wkt"],None)

Variable,Value
a_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
isASimple,true
o_wkt,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon ((2.47217 47.52403, 2.40693 47.49713, 2.44607 47.43679, 2.57104 47.48553, 2.39251 47.44793, 2.47217 47.52403))"
isOSimple,false


Map(center=[47.47714370265989, 2.4756124648711317], controls=(ZoomControl(options=['position', 'zoom_in_text',…

### [geof:maxX](http://www.opengis.net/def/function/geosparql/maxX) Function

Retrieves the maximum X coordinate of a GeoSPARQL geometry as [xsd:double](http://www.w3.org/2001/XMLSchema#double).
The X coordinate axis is defined by the CRS of the respective geometry.

In [17]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?maxX
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:maxX(?aLiteral) as ?maxX)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
maxX,-83.2


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:maxY](http://www.opengis.net/def/function/geosparql/maxY) Function

Retrieves the maximum Y coordinate of a GeoSPARQL geometry as [xsd:double](http://www.w3.org/2001/XMLSchema#double). The Y coordinate axis is defined by the CRS of the respective geometry.

In [18]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?maxY
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:maxY(?aLiteral) as ?maxY)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
maxY,34.5


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:maxZ](http://www.opengis.net/def/function/geosparql/maxZ) Function

Retrieves the maximum Z coordinate of a GeoSPARQL geometry as [xsd:double](http://www.w3.org/2001/XMLSchema#double). The Z coordinate axis is defined by the CRS of the respective geometry.

In [19]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?maxZ
WHERE {
  my:NExactGeom geo:asWKT ?literal .
  BIND(geof:maxZ(?literal) AS ?maxZ)
}
"""),["literal"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon Z((-77.089005 38.913574 1,-77.029953 38.913574 2,-77.029953 38.886321 2,-77.089005 38.886321 1,-77.089005 38.913574 1))"
maxZ,2.0


Map(center=[38.8999475, -77.05947900000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

### [geof:metricArea](http://www.opengis.net/def/function/geosparql/metricArea) Function

Returns the area of a GeoSPARQL geometry in squaremeters as [xsd:double](http://www.w3.org/2001/XMLSchema#double).

In [20]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?marea
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:metricArea(?aLiteral) as ?marea)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
marea,2400116828.6431704


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:metricBuffer](http://www.opengis.net/def/function/geosparql/metricBuffer) Function

In [21]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?buffer
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:metricBuffer(?aLiteral,"5.0"^^xsd:double) as ?buffer)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
buffer,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> POLYGON ((-9306314.43031767 4042237.4996761675, -9306314.43031767 4096139.0404472337, -9306314.334244072 4096140.0158988438, -9306314.049715333 4096140.9538643956, -9306313.587665731 4096141.818298399, -9306312.965851575 4096142.5759811397, -9306312.208168834 4096143.197795295, -9306311.343734832 4096143.6598448963, -9306310.405769281 4096143.9443736356, -9306309.43031767 4096144.0404472337, -9261781.634000361 4096144.0404472337, -9261780.65854875 4096143.9443736356, -9261779.720583199 4096143.6598448963, -9261778.856149197 4096143.197795295, -9261778.098466456 4096142.5759811397, -9261777.4766523 4096141.818298399, -9261777.014602698 4096140.9538643956, -9261776.730073959 4096140.0158988438, -9261776.634000361 4096139.0404472337, -9261776.634000361 4042237.4996761675, -9261776.730073959 4042236.5242245574, -9261777.014602698 4042235.5862590056, -9261777.4766523 4042234.721825002, -9261778.098466456 4042233.9641422615, -9261778.856149197 4042233.342328106, -9261779.720583199 4042232.880278505, -9261780.65854875 4042232.5957497656, -9261781.634000361 4042232.4996761675, -9306309.43031767 4042232.4996761675, -9306310.405769281 4042232.5957497656, -9306311.343734832 4042232.880278505, -9306312.208168834 4042233.342328106, -9306312.965851575 4042233.9641422615, -9306313.587665731 4042234.721825002, -9306314.049715333 4042235.5862590056, -9306314.334244072 4042236.5242245574, -9306314.43031767 4042237.4996761675))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:metricDistance](http://www.opengis.net/def/function/geosparql/metricDistance) Function

This function returns a distance between GeoSPARQL geometries in meters as [xsd:double](http://www.w3.org/2001/XMLSchema#double).

In [22]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?mdistance
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:metricDistance(?cLiteral, ?fLiteral) as ?mdistance)
}
"""),["cLiteral","fLiteral"],None)

Variable,Value
cLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
fLiteral,<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Point(-83.4 34.4)
mdistance,22263.89815865457


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:metricLength](http://www.opengis.net/def/function/geosparql/metricLength) Function

Returns the length of a given geometry in meters as [xsd:double](http://www.opengis.net/ont/geosparql#double)

In [23]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?mlength
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:metricLength(?aLiteral) as ?mlength)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
mlength,196858.6741767507


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:minX](http://www.opengis.net/def/function/geosparql/minX) Function

Retrieves the minimum X coordinate of a GeoSPARQL geometry as [xsd:double](http://www.w3.org/2001/XMLSchema#double). The X coordinate axis is defined by the CRS of the respective geometry.

In [24]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?minX
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:minX(?aLiteral) as ?minX)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
minX,-83.6


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:minY](http://www.opengis.net/def/function/geosparql/minY) Function

Retrieves the minimum Y coordinate of a GeoSPARQL geometry as [xsd:double](http://www.w3.org/2001/XMLSchema#double). The Y coordinate axis is defined by the CRS of the respective geometry.

In [25]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?minY
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:minY(?aLiteral) as ?minY)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
minY,34.1


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:minZ](http://www.opengis.net/def/function/geosparql/minZ) Function

Retrieves the minimum Z coordinate of a GeoSPARQL geometry as [xsd:double](http://www.w3.org/2001/XMLSchema#double). The Z coordinate axis is defined by the CRS of the respective geometry.

In [26]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?minZ
WHERE {
  my:NExactGeom geo:asWKT ?literal .
  BIND(geof:minZ(?literal) AS ?minZ)
}
"""),["literal"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon Z((-77.089005 38.913574 1,-77.029953 38.913574 2,-77.029953 38.886321 2,-77.089005 38.886321 1,-77.089005 38.913574 1))"
minZ,1.0


Map(center=[38.8999475, -77.05947900000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

### [geof:numGeometries](http://www.opengis.net/def/function/geosparql/numGeometries) Function

Returns the number of geometries included in the given geometry as [xsd:integer](http://www.w3.org/2001/XMLSchema#integer)

In [27]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?numGeoms
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:numGeometries(?literal) AS ?numGeoms)
}
"""),["literal"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
numGeoms,1


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:spatialDimension](http://www.opengis.net/def/function/geosparql/spatialDimension) Function

Returns the spatial dimension of a given geometry as [xsd:integer](http://www.opengis.net/ont/geosparql#integer)

In [28]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?spatialDimension
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:spatialDimension(?literal) AS ?spatialDimension)
}
"""),["literal"],None)

Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
spatialDimension,2


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:transform](http://www.opengis.net/def/function/geosparql/transform) Function

Transforms a given geometry from its current CRS to a CRS given by a parameter and returns it in the literal format of the input geometry

In [29]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?transformed
WHERE {
  my:AExactGeom geo:asWKT ?literal .
  BIND(geof:transform(?literal,"http://www.opengis.net/def/crs/EPSG/0/4326") AS ?transformed)
}
"""),["literal"],None)

TRANSFORM FUNCTION
GEOM: POLYGON ((34.1 -83.6, 34.1 -83.2, 34.5 -83.2, 34.5 -83.6, 34.1 -83.6))
AS LITERAL: <http://www.opengis.net/def/crs/EPSG/0/4326> POLYGON ((34.1 -83.6, 34.1 -83.2, 34.5 -83.2, 34.5 -83.6, 34.1 -83.6))


Variable,Value
literal,"<http://www.opengis.net/def/crs/OGC/1.3/CRS84> Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
transformed,"<http://www.opengis.net/def/crs/EPSG/0/4326> POLYGON ((34.1 -83.6, 34.1 -83.2, 34.5 -83.2, 34.5 -83.6, 34.1 -83.6))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…